## Reading text-data into Python

In [ ]:
# read the file in the project directory
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("total number of character: ", len(raw_text))
# print first 100
print(raw_text[:99])

## Split text to obtain a list of tokens
- split the text on all types of punctuations, whitespace(\s), commas, periods, question marks, quotation marks, double-dashes

In [ ]:
import re
# text = "Hello, world. This is a test. Hello, world. This is a test."
text = raw_text

# Because of (), these separators are also included in the result.
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) 

# .strip() removes whitespace from both ends and store in [] 
# if item.strip() decides whether to keep the item, don't keep "<empty_str>"
preprocessed = [item.strip() for item in preprocessed if item.strip()] 

# print(preprocessed)
print(f"the number of tokens in preprocessed text: {len(preprocessed)}")

## Converting tokens into token IDs
- convert these tokens from a __string to an integer__ representation to produce the token IDs
- individual tokens are then sorted alphabetically, and duplicate tokens are removed
- each sorted token is then mapped to an integer.

In [ ]:
# sorted set of tokens
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"length of sorted set of tokens: {vocab_size}")

# dictionary of token and corresponding token id
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i>=10:
        break
        

### Complete tokenizer class with encode and decode method
- encode method: splits text into tokens and carries out the string string-to-integer mapping to produce token IDs via the vocabulary
- decode method: carries out reverse integer-to-string mapping to convert the token IDs back into text.
  

In [ ]:
class SimpleTokenizerV1:

    # constructor takes a vocab, dictionary of token -> ID
    def __init__(self, vocab):
        # Stores the vocabulary as a class attribute for access in the encode and decode methods
        self.str_to_int = vocab 
        # Creates an inverse vocubalary ID -> token
        self.int_to_str = {i:s for s,i in vocab.items()} 

    # process input text into token
    def encode(self, text):
        preprocessed = re.split(r'([,.?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # convert token IDs back into text
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids]) # "separator".join(list) 
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text) # r'\1' captured punctuation, replaces " !" -> "!"
        return text
    

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride."""

# text = "Hello, do you like tea?" # KeyError, Hello was not used in vocab 

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

### Adding special context tokens
- to handle unkown words, tokenizer needs to be modifies ie. extended vocabulary with additional special tokens
- <|endoftext|> tokens act as markers, signaling the start or end of a particular segment, allowing for more effective processing and understanding by the LLM
- 

In [ ]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer,token in enumerate(all_tokens)}
print("the new vocabulary size: ",len(vocab.items()))
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

## tokenizer V2 to handle unkown words

In [ ]:
class SimpleTokenizerV2:
    def __init__(self,vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self,text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)',text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [item if item in self.str_to_int
                        else "<|unk|>" for item in preprocessed]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
        return text
                    

In [ ]:
# <\endoftext\> token to concatenate from two unrelated sentences
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1,text2))
print(text)

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

## Byte pair ecoding used by GPT models
- breaks words down into subword units
- implemented here using open source library __tiktoken__
- The BPE tokenizer can handle any unknown word.

In [ ]:
!pip install tiktoken

In [ ]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
"Hello, do you like tea? <|endoftext|> In the sunlit terraces"
"of someunknownPlace."
)
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
strings = tokenizer.decode(integers)
print(strings)

## Data sampling with a sliding window 
- the input–target pairs required for training an LLM
- LLMs are pretrained by predicting the next word in a text,
- tokenize the whole "The Verdict" using BPE tokenizer.
- to create input-target pairs for next word prediction, create two variables x and y, x contains input tokens and y contains target.

In [ ]:
with open("../the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    enc_text = tokenizer.encode(raw_text)
# print(len(enc_text))

enc_sample = enc_text[50:]

# context size determines number of tokens are included in the input
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:       {y}")

In [ ]:
# By processing the inputs along with the targets, which are the inputs shifted by one position,
# create the next-word prediction tasks
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # left of arrow is ? LLM gets and right is ? supposed to predict 
    print(context, "---->", desired) 
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))
    print("\n")

### PyTorch built-in Dataset and DataLoader classes

In [ ]:
!pip install torch

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataSetV1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt)

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i+1: i+max_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx], self.target_ids[idx]
    